# Flight Delay Prediction — Phase 2 (Deep Learning)

## 1. Setup

In [1]:
import os
# PyTorch and XGBoost each ship their own OpenMP library; on macOS they crash if both start
# multi-threaded OpenMP. One OpenMP thread avoids this (PyTorch still uses 4 threads below).
os.environ["OMP_NUM_THREADS"] = "1"
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from nycflights13 import flights
from sklearn.metrics import (average_precision_score, roc_auc_score, brier_score_loss,
                             precision_recall_curve, roc_curve)

sns.set_style("whitegrid")
os.makedirs("figures", exist_ok=True)

RANDOM_STATE = 42
torch.set_num_threads(4)   # CPU training is fast enough for this data and gives repeatable results
print("PyTorch", torch.__version__)

PyTorch 2.15.0


## 2. Data and features (same pipeline as Phase 1)

In [2]:
df = flights.copy()

# Drop cancelled / diverted flights (no arrival delay), build the target and a date column
df = df.dropna(subset=["arr_delay"]).copy()
df["delayed"] = (df["arr_delay"] >= 15).astype(int)
df["date"] = pd.to_datetime(df[["year", "month", "day"]])

# Remove everything only known after departure (data leakage) and unused ID / duplicate columns
leaky_columns = ["dep_time", "dep_delay", "arr_time", "arr_delay", "air_time"]
df = df.drop(columns=leaky_columns + ["tailnum", "flight", "time_hour", "year", "day", "hour", "minute"])

print("Flights:", len(df), " delayed share:", round(df["delayed"].mean(), 3))

Flights: 327346  delayed share: 0.245


In [3]:
# Calendar and schedule features with cyclic (sin/cos) encoding
df["day_of_week"] = df["date"].dt.dayofweek
df["dep_hour"] = df["sched_dep_time"] // 100
df["arr_hour"] = df["sched_arr_time"] // 100
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

def add_cyclic(df, col, period):
    df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
    df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)

add_cyclic(df, "dep_hour", 24)
add_cyclic(df, "arr_hour", 24)
add_cyclic(df, "day_of_week", 7)

# Airport congestion, counted on the FULL schedule (including flights that were later cancelled)
schedule = flights.copy()
schedule["date"] = pd.to_datetime(schedule[["year", "month", "day"]])
schedule["dep_hour"] = schedule["sched_dep_time"] // 100
hour_counts = schedule.groupby(["date", "origin", "dep_hour"]).size().rename("origin_hour_flights")
day_counts = schedule.groupby(["date", "origin"]).size().rename("origin_day_flights")
df = df.join(hour_counts, on=["date", "origin", "dep_hour"]).join(day_counts, on=["date", "origin"])

In [4]:
def add_past_delay_rate(df, col, window=7):
    """Share of delayed flights for each value of `col` over the previous `window` days (today excluded)."""
    delayed_per_day = df.pivot_table(index="date", columns=col, values="delayed", aggfunc="sum", fill_value=0)
    flights_per_day = df.pivot_table(index="date", columns=col, values="delayed", aggfunc="count", fill_value=0)
    past_delayed = delayed_per_day.rolling(window, min_periods=1).sum().shift(1)
    past_flights = flights_per_day.rolling(window, min_periods=1).sum().shift(1)
    rate = past_delayed / past_flights.replace(0, np.nan)
    rate = rate.stack().rename(f"{col}_delay_rate_7d").reset_index()
    return df.merge(rate, on=["date", col], how="left")

df["route"] = df["origin"] + "-" + df["dest"]
for col in ["carrier", "origin", "dest", "route"]:
    df = add_past_delay_rate(df, col)

numeric_features = ["is_weekend", "day_of_week_sin", "day_of_week_cos",
                    "dep_hour_sin", "dep_hour_cos", "arr_hour_sin", "arr_hour_cos", "distance",
                    "origin_hour_flights", "origin_day_flights",
                    "carrier_delay_rate_7d", "origin_delay_rate_7d", "dest_delay_rate_7d", "route_delay_rate_7d"]
history_features = [c for c in numeric_features if "delay_rate" in c]
categorical_features = ["carrier", "origin", "dest"]

assert not set(leaky_columns) & set(numeric_features + categorical_features)

## 3. Train / validation / test split by date

In [5]:
train = df[df["month"] <= 8].reset_index(drop=True)          # Jan-Aug
val = df[df["month"].isin([9, 10])].reset_index(drop=True)    # Sep-Oct
test = df[df["month"] >= 11].reset_index(drop=True)           # Nov-Dec
assert train["date"].max() < val["date"].min() and val["date"].max() < test["date"].min()

y_train, y_val, y_test = train["delayed"].values, val["delayed"].values, test["delayed"].values

pd.DataFrame({"flights": [len(train), len(val), len(test)],
              "delayed_share": [y_train.mean(), y_val.mean(), y_test.mean()]},
             index=["train", "validation", "test"]).round(3)

,flights,delayed_share
train,217727,0.264
validation,55628,0.157
test,53991,0.258
